In [1]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import requests
import pandas as pd
import joblib

In [2]:
def create_embedding(text_list):
    r = requests.post("http://localhost:11434/api/embed",json={
        "model":"bge-m3",
        "input":text_list
        # "prompt":text
    })
    embedding = r.json()['embeddings']
    return embedding

def inference(prompt):
    r = requests.post("http://localhost:11434/api/generate",json={
        #"model" : "deepseek-r1"
        "model":"llama3.2",
        "prompt":prompt,
        "stream" : False
        # "prompt":text
    })
    response = r.json()
    print(response)
    return response

df = joblib.load('embedding.joblib')

incoming_query = input("Ask a Question : ")
question_embedding = create_embedding([incoming_query])[0]

similarity = cosine_similarity(np.vstack(df['embedding']),[question_embedding]).flatten()
print(similarity)
top_result = 3
max_indx = similarity.argsort()[::-1][0:top_result]
print(max_indx)
new_df = df.loc[max_indx]
print(new_df[['title','number','text']])


[0.64030449 0.5656805  0.46283583 0.41550219 0.30939625 0.57188999
 0.35423636 0.4445302  0.59760033 0.32744491 0.41337275 0.30713242
 0.34037426 0.32929233 0.36272641 0.35335078 0.44577027 0.35734345
 0.28330789 0.59633251 0.3469331  0.75092544 0.57542586 0.46161326
 0.36831459 0.38630379 0.37657549 0.4609536  0.37851083 0.32968205
 0.3762504  0.44575553 0.37319915 0.33981414 0.39708967 0.35539895
 0.34405325 0.3541126  0.36029381 0.45663322 0.40854561 0.36859705
 0.28006625 0.35685121 0.35921326 0.30479314 0.33133898 0.42247968
 0.37632146 0.3208374  0.340867   0.35687589 0.36831619 0.36465052
 0.36856592 0.31593811 0.38563149 0.39630178 0.38080625 0.28758018
 0.36408984 0.3331384  0.2993947  0.31594011 0.37254332 0.37381313
 0.35504417 0.35120694 0.3872108  0.32288083 0.42871406 0.5741485
 0.56110954 0.46307723 0.43714232 0.38955115 0.39395008 0.49326143
 0.48865538 0.38867062 0.34707812 0.62519408 0.52929013 0.36232044
 0.53487891 0.38067688 0.34877052 0.4158363  0.40326511 0.39520

In [3]:
prompt = f'''I am teaching web developement  using Sigma web developement course. Here are video
subtitle chunks containing video title,video number,start time in seconds,end time in seconds, the text
at that time :

{new_df[["title","number","start","end","text"]].to_json(orient = "records")}
-------------------------------------------------------
"{incoming_query}"
User asked this question related to the video chunks, you have to answer where and how much 
content is taught in which video (in which video and at what timeshamp) and guide the user to go
to that particular video. If user asks unrelated question, tell him that you can only answer 
question related to the course.
'''
with open("prompt.txt","w") as f:
    f.write(prompt)

response = inference(prompt)["response"]
print(response)

{'model': 'llama3.2', 'created_at': '2026-06-02T09:27:40.8683682Z', 'response': 'Based on the provided subtitle chunks, it appears that the inline elements are taught in Video 8 of the Sigma Web Development course.\n\nSpecifically:\n\n* The first mention of "inline" is at time 79.24 seconds, where the instructor explains what inline elements are and why they\'re useful.\n* The second mention of "inline" is at time 371.24 seconds, where the instructor lists some examples of inline elements.\n* The third mention of "inline" is at time 616.24 seconds, where the instructor compares and contrasts inline and block elements.\n\nSo, to answer your question, inline elements are taught in Video 8 of the Sigma Web Development course, starting from approximately 79.24 seconds until 618.24 seconds.', 'done': True, 'done_reason': 'stop', 'context': [128006, 9125, 128007, 271, 38766, 1303, 33025, 2696, 25, 6790, 220, 2366, 18, 271, 128009, 128006, 882, 128007, 271, 40, 1097, 12917, 3566, 2274, 1133, 

In [4]:
with open("response.txt",'w') as f:
    f.write(response)